# 第 13 章 操作変数の検定: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/13-iv-tests.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0                       # 乱数の種
n = 500                        # 標本の大きさ
invalid_levels = [0.0, 0.1, 0.3]  # 操作変数 z2 が誤差と相関する度合い
endogeneity = 0.5              # Wu-Hausman 検定で調べる内生性の強さ
significance = 0.05            # 有意水準
repetitions = 300              # 標本を取り直す回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## Sargan 検定と Wu-Hausman 検定

In [ ]:
import numpy as np

from lib.chapter13.iv_tests import (
    draw_overid_sample,
    sargan_rejection_rate,
    sargan_test,
    wu_hausman_rejection_rate,
)

sample = draw_overid_sample(n, invalid_levels[-1], np.random.default_rng(seed))
test = sargan_test(sample["y"], sample[["x"]], sample[["z1", "z2"]])
print(f"1 つの標本の Sargan 検定（invalid={invalid_levels[-1]}）: "
      f"統計量 {test.statistic:.3f}、p 値 {test.p_value:.4f}")

rates = {
    f"Invalid {invalid}": sargan_rejection_rate(
        n, invalid, repetitions, significance, np.random.default_rng(seed)
    )
    for invalid in invalid_levels
}
for name, rate in rates.items():
    print(f"Sargan の棄却率 {name}: {rate:.3f}")
wu = wu_hausman_rejection_rate(
    n, endogeneity, repetitions, significance, np.random.default_rng(seed)
)
print(f"Wu-Hausman の棄却率（内生性 {endogeneity}）: {wu:.3f}")

## 図: Sargan 検定の棄却率

In [ ]:
from lib.chapter13.plot import plot_rejection_rates

plot_rejection_rates(rates, significance)

## 試してみよう

- `invalid_levels` の 0.0・0.1・0.3 で、Sargan 検定の棄却率は 0.053・0.390・1.000
  です。操作変数が外生（0.0）なら有意水準 5% に近く、誤差と相関するほど、見抜けます
- `n` を 2000 にすると、`invalid` が 0.1 の棄却率は 0.390 から 0.930
  に上がります（標本が大きいほど、小さな不備も、見抜けます）
- `endogeneity` を 0.0 にすると、Wu-Hausman 検定の棄却率は 0.027 に下がり、有意水準
  5% より小さくなります（内生性がなければ、棄却しません）